Link to my folder where it is pulling the datasets from. 
Coordinates:
Lat: 33.69 , 34.51
Longitude: -120.52 , -118.98 

In [ ]:
import os
from pathlib import Path
if Path.cwd().name == "Analysis":
    os.chdir("..")
print(Path.cwd())

This connects to the SQL database and I choose duckDB because it can easily transfer from csv files and the dialect is close to PostgreSQL

In [ ]:
import duckdb
con = duckdb.connect("ocean.duckdb")
con.sql("SHOW TABLES").df()

In [ ]:
con.execute(open("ocean.sql").read())
con.execute("CHECKPOINT")
con.sql("SELECT view_name FROM duckdb_views() WHERE NOT internal").show()

In [ ]:
con.sql("CALL start_ui()")

--- run cells 1 and 4 till here once the database has been set up



This pulls in the SST_ MUR datasets from 2018 through 2026. 
It puts it together based on the timestamp, lat and long, and then pulls it from the sst_mur datasets. 

In [ ]:
con.sql("""
CREATE OR REPLACE VIEW sst AS
SELECT CAST(time AS TIMESTAMP) AS ts,
       CAST(latitude AS DOUBLE) AS lat,
       CAST(longitude AS DOUBLE) AS lon,
       CAST(analysed_sst AS DOUBLE) AS sst_c
FROM 'datasets/SST_MUR_*.csv'
WHERE TRY_CAST(latitude AS DOUBLE) IS NOT NULL
""")

This is the chlorophyll through dineof that already has a model to predict where the chlorophyll would be based on DINEOF; which estimates chlorophyll on cloudy days based on patterns over space and time.

In [ ]:
con.sql("""
CREATE OR REPLACE VIEW chl_dineof AS
SELECT CAST(time AS TIMESTAMP) AS ts,
       CAST(latitude AS DOUBLE) AS lat,
       CAST(longitude AS DOUBLE) AS lon,
       CAST(chlor_a AS DOUBLE) AS chl_filled
FROM 'datasets/chlorophyll_*_dineof.csv'
WHERE TRY_CAST(latitude AS DOUBLE) IS NOT NULL
""")

This would include the observed through satellite chlorophyll - however there is limited data due to cloud coverage so there are a lot of null values 

In [ ]:
con.sql("""
CREATE OR REPLACE VIEW chl_erd AS
SELECT CAST(time AS TIMESTAMP) AS ts,
       CAST(latitude AS DOUBLE) AS lat,
       CAST(longitude AS DOUBLE) AS lon,
       CAST(chla AS DOUBLE) AS chl_obs
FROM 'datasets/erd_chlorophyll_*.csv'
WHERE TRY_CAST(latitude AS DOUBLE) IS NOT NULL
""")

Next is Pacific Decadal Oscillation (PDO) - this is what looks at the patterns between the extreme phases associated with El Niño. Something that influences water temperatures of the ocean which could contribute to the overall health of the ocean. 

below this dataset did not have long and lat so it so I didn't need to run cast on it. Looked into the UNPIVOT which turns the 12 columns into 12 rows and then reads the moonth as a date. 

In [ ]:
con.sql("""
CREATE OR REPLACE VIEW pdo AS
SELECT make_date(Year, month(strptime(month_name, '%b')), 1) AS date,
       pdo
FROM (
    UNPIVOT (SELECT * FROM 'datasets/pdo_data.csv')
    ON Jan, Feb, Mar, Apr, May, Jun, Jul, Aug, Sep, Oct, Nov, Dec
    INTO NAME month_name VALUE pdo
)
""")

This is to checks till when I have my dataset so the first date is 1850 and last is 2026 

In [ ]:
con.sql("SELECT min(date), max(date), count(*) FROM pdo").df()

Next to implement the Land_weather datasets from Santa Barbara and Ventura to look at stations that are measuring levels of run off in addition to percipitation run off. This is in case I wanted to correlate how much of the rain could result in a bloom for the chlorophyll depending on terrestial run off. 

In addition for the CAST(PRCP AS DOUBLE)* 25.4 as prcp - the original prcp is in inches rain now I am moving it to mm 

In [ ]:
con.sql("""
CREATE OR REPLACE VIEW rain AS
SELECT STATION AS station, NAME AS name,
       CAST(LATITUDE AS DOUBLE) AS lat,
       CAST(LONGITUDE AS DOUBLE) AS lon,
       CAST(DATE AS DATE) AS date,
       CASE WHEN PRCP = -9999 THEN NULL ELSE CAST(PRCP AS DOUBLE) * 25.4 END AS prcp,
       CASE WHEN filename LIKE '%SB%' THEN 'santa_barbara' ELSE 'ventura' END AS region
FROM read_csv_auto('datasets/land_weather_*.csv', union_by_name = TRUE, filename = TRUE)
WHERE PRCP IS NOT NULL
""")

In [ ]:
con.sql("SELECT min(date), max(date), min(prcp), max(prcp), mean(prcp), count(*) FROM rain").df()

Next include the glider data that has DOXY from ventura county to measure the levels of disolved oxygen 

In [ ]:
con.sql("""
CREATE OR REPLACE VIEW glider AS
SELECT CAST(mission AS INTEGER) AS mission,
       CAST(time AS TIMESTAMP) AS ts,
       CAST(latitude AS DOUBLE) AS lat,
       CAST(longitude AS DOUBLE) AS lon,
       CAST(depth AS DOUBLE) AS depth,
       CASE WHEN doxy = 'NaN' THEN NULL ELSE TRY_CAST(doxy AS DOUBLE) END AS doxy,
       CASE WHEN temperature = 'NaN' THEN NULL ELSE TRY_CAST(temperature AS DOUBLE) END AS temperature,
       CASE WHEN salinity = 'NaN' THEN NULL ELSE TRY_CAST(salinity AS DOUBLE) END AS salinity,
       CASE WHEN chlorophyll = 'NaN' THEN NULL ELSE TRY_CAST(chlorophyll AS DOUBLE) END AS chl_glider
FROM read_csv_auto('datasets/binnedCUGN80_52e6_d0ac_a341.csv', all_varchar=true)
WHERE TRY_CAST(latitude AS DOUBLE) IS NOT NULL
""")


For this dataset they used the timestamp instead of date so now instead of running date we run ts.

In [ ]:
con.sql("SELECT min(ts), max(ts), min(doxy), max(doxy), mean(doxy), count(*) FROM glider").df()


In [ ]:
con.sql("""
SELECT ts, lat, lon, depth, doxy, temperature, chl_glider
FROM glider
WHERE doxy IS NOT NULL
ORDER BY doxy DESC
LIMIT 10
""").df()

In [ ]:
con.sql("SELECT depth, doxy, temperature, chl_glider FROM glider WHERE ts = '2025-06-01 19:47:40' ORDER BY depth").df()

Next Kelp Data 
-----
Sea urchins
Purple Urchin Scientific name: Strongylocentrotus purpuratus
Red Urchin - Mesocentrotus franciscanus
Crowned Sea Urchin - Centrostephanus coronatus
White Sea Urchin : Lytechinus pictus
---
Predators
Sunflower Sea Star - Pycnopodia helianthoides (predator of sea urchin with illness)
California Spiny Lobster- Panulirus interruptus (another predator)

---
Kelp
easier to find through Taxon_family = Lessoniaceae

In [ ]:
con.sql("""
CREATE OR REPLACE VIEW reef AS
SELECT SITE AS site,
       TRANSECT AS transect,
       CAST(DATE AS DATE) AS date,
       YEAR AS year, MONTH AS month,
       SCIENTIFIC_NAME AS species,
       COMMON_NAME AS common_name,
       NULLIF(DENSITY, -99999) AS density,
       NULLIF(PERCENT_COVER, -99999) AS percent_cover,
       NULLIF(WM_GM2, -99999) AS wet_biomass
FROM read_csv_auto('datasets/SBS_All_Species_Biomass_at_transect_20260311.csv')
""")

to see per site per year data however best to go through the names I care for individually

In [ ]:
con.sql("""
PIVOT (SELECT year, common_name, density FROM reef
       WHERE species IN ('Strongylocentrotus purpuratus','Mesocentrotus franciscanus',
                         'Centrostephanus coronatus','Lytechinus pictus'))
ON common_name USING round(avg(density), 4) GROUP BY year ORDER BY year
""").df()


In [ ]:
con.sql("""
PIVOT (SELECT year, common_name, density FROM reef
       WHERE species IN ('Strongylocentrotus purpuratus','Mesocentrotus franciscanus',
                         'Pycnopodia helianthoides','Panulirus interruptus','Macrocystis pyrifera'))
ON common_name USING round(avg(density), 4) GROUP BY year ORDER BY year
""").df()

In [ ]:
con.sql("SHOW TABLES").df()

This is the dataset for the kelp:

In [ ]:

con.sql("CREATE OR REPLACE VIEW kelp AS SELECT * FROM 'datasets/kelp_landsat.parquet'")

Cuti index for upwelling - originally build it in python document but then uploaded the rest in a jupyter notebook

In [ ]:

con.sql("""
CREATE OR REPLACE VIEW cuti AS
SELECT CAST(time AS TIMESTAMP) AS ts,
       CAST(latitude AS DOUBLE) AS lat,
       CAST(CUTI AS DOUBLE) AS cuti
FROM 'datasets/erdCUTIdaily_c010_b954_09c2.csv'
WHERE TRY_CAST(latitude AS DOUBLE) IS NOT NULL
""")
